# SeoulMate 벡터DB 구축 (restaurant)

`data/db_seed/restaurant/`의 시드 CSV를 PostgreSQL + pgvector에 적재하고 임베딩을 생성합니다.
**현재 운영 DB와 동일한 테이블 구조**(언어별 ko/en, 단일 `menu_name`, `menu_embedding` 포함)를 만듭니다.

**한/영 둘 다 만들려면**: 셀 1의 `LANG`을 `"ko"` → `"en"`으로 바꿔 **두 번 실행**하세요.
(식당·리뷰·메뉴·임베딩 모두 언어별 테이블이라 각 언어마다 전부 적재됩니다.)

### 생성 테이블 (LANG별)
- `restaurant_{lang}` (카카오 컬럼 16개 포함) · `restaurant_review_{lang}`
- `restaurant_menu_{lang}` — 단일 `menu_name` 컬럼
- `restaurant_embedding_{lang}` (이름+카테고리+설명+대표메뉴 임베딩)
- `review_embedding_{lang}` (리뷰별, `review_id` FK)
- `menu_embedding_{lang}` (메뉴명 임베딩, `menu_id`+`restaurant_id`)

### 준비물 (`data/db_seed/restaurant/`)
`restaurant_ko.csv`, `restaurant_en.csv`,
`restaurant_menu_ko.csv`, `restaurant_menu_en.csv`,
`restaurant_review_ko.csv`, `restaurant_review_en.csv`

시드 폴더는 자동 탐색합니다. 못 찾으면 `SEED_DIR` 환경변수로 절대경로를 지정하세요.

> ⚠️ `RESET_DB=True`면 해당 언어 테이블을 **DROP 후 재생성**합니다. 운영 DB에 실행하면 데이터가 지워지니 주의하세요.

## 1. 설정

In [ ]:
import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from openai import OpenAI

# ─────────────── 여기만 바꾸면 됩니다 ───────────────
LANG = "ko"          # "ko" 또는 "en"  ← 두 번 실행
RESET_DB = True      # True면 해당 언어 테이블을 DROP 후 재생성
# ──────────────────────────────────────────────────

DB_PARAMS = {
    "host": os.environ.get("DB_HOST", "localhost"),
    "port": int(os.environ.get("DB_PORT", "5433")),
    "user": os.environ.get("DB_USER", "seoulmate"),
    "password": os.environ.get("DB_PASSWORD"),
    "database": os.environ.get("DB_NAME", "seoulmate"),
}

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM   = 1536
BATCH       = 200            # 임베딩 배치 크기
MENU_TOP_N  = 8              # 임베딩 텍스트에 넣을 대표메뉴 개수


# ─── 시드 CSV 위치: backend/data/db_seed/restaurant/ ───
# 노트북을 어디서 실행하든 시드 폴더를 찾도록 여러 후보 경로를 시도한다.
# SEED_DIR 환경변수로 직접 지정할 수도 있다.
def _find_seed_dir() -> Path:
    if os.environ.get("SEED_DIR"):
        return Path(os.environ["SEED_DIR"])
    cwd = Path.cwd()
    candidates = [
        cwd / "data" / "db_seed" / "restaurant",                       # cwd == backend/
        cwd / "backend" / "data" / "db_seed" / "restaurant",           # cwd == repo 루트
    ]
    # cwd == backend/vector_db/restaurant/ 처럼 하위에서 실행하는 경우
    for parent in cwd.parents:
        candidates.append(parent / "data" / "db_seed" / "restaurant")
        if parent.name == "backend":
            break
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        "시드 폴더(data/db_seed/restaurant)를 찾지 못했습니다. "
        "SEED_DIR 환경변수로 절대경로를 지정하세요."
    )


BASE = _find_seed_dir()
REST_CSV = BASE / f"restaurant_{LANG}.csv"
REV_CSV  = BASE / f"restaurant_review_{LANG}.csv"
MENU_CSV = BASE / f"restaurant_menu_{LANG}.csv"

# 현재 DB와 동일하게 모든 테이블을 언어별로 둔다(메뉴 포함).
T_REST  = f"restaurant_{LANG}"
T_REV   = f"restaurant_review_{LANG}"
T_MENU  = f"restaurant_menu_{LANG}"
T_REMB  = f"restaurant_embedding_{LANG}"
T_VEMB  = f"review_embedding_{LANG}"
T_MEMB  = f"menu_embedding_{LANG}"

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print(f"LANG={LANG} | SEED_DIR={BASE}")
print(f"테이블: {T_REST}, {T_REV}, {T_MENU}, {T_REMB}, {T_VEMB}, {T_MEMB}")

## 2. CSV 로드

In [ ]:
rest = pd.read_csv(REST_CSV, encoding="utf-8-sig", dtype={"postal_code": str})
rev  = pd.read_csv(REV_CSV,  encoding="utf-8-sig")
menu = pd.read_csv(MENU_CSV, encoding="utf-8-sig")

# NaN → None (psycopg2가 NULL로 넣도록)
rest = rest.replace({np.nan: None})
rev  = rev.replace({np.nan: None})
menu = menu.replace({np.nan: None})

print(f"식당 {len(rest):,}행 x {len(rest.columns)}컬럼")
print(f"리뷰 {len(rev):,}행")
print(f"메뉴 {len(menu):,}행")
print()
print(f"  description        {rest['description'].notna().sum():4}")
print(f"  description_kakao  {rest['description_kakao'].notna().sum():4}")
print(f"  category_kakao     {rest['category_kakao'].notna().sum():4}")
print(f"  hours              {rest['hours'].notna().sum():4}")
print(f"  menu_price_median  {rest['menu_price_median'].notna().sum():4}")

## 3. 테이블 생성 (DDL)

`RESET_DB=True`면 기존 테이블을 지우고 새로 만듭니다.

**`review_embedding`의 FK가 `review_id`** 입니다 — 어느 리뷰에서 만든 임베딩인지 추적할 수 있어요.

In [ ]:
DDL = f"""
CREATE EXTENSION IF NOT EXISTS vector;
CREATE EXTENSION IF NOT EXISTS pg_trgm;

CREATE TABLE IF NOT EXISTS {T_REST} (
    id                  BIGINT PRIMARY KEY,
    name                TEXT,
    phone               TEXT,
    address             TEXT,
    postal_code         TEXT,
    lat                 DOUBLE PRECISION,
    lng                 DOUBLE PRECISION,
    category            TEXT,
    hours               TEXT,
    description         TEXT,
    image               TEXT,
    link                TEXT,
    review_count        INT,
    rating              DOUBLE PRECISION,
    -- 카카오 병합 컬럼
    category_kakao      TEXT,
    description_kakao   TEXT,
    last_order          TEXT,
    kakao_place_id      TEXT,
    kakao_place_url     TEXT,
    menu_price_min      INT,
    menu_price_median   INT,
    menu_count          INT,
    has_parking         BOOLEAN,
    has_group_seating   BOOLEAN,
    has_private_room    BOOLEAN,
    has_baby_chair      BOOLEAN,
    has_kids_menu       BOOLEAN,
    allows_pets         BOOLEAN,
    has_disabled_access BOOLEAN,
    hours_source        TEXT
);

CREATE TABLE IF NOT EXISTS {T_REV} (
    id            BIGINT PRIMARY KEY,
    restaurant_id BIGINT REFERENCES {T_REST}(id),
    rating        DOUBLE PRECISION,
    content       TEXT
);

-- 메뉴는 현재 DB와 동일하게 언어별 테이블 + 단일 menu_name 컬럼
CREATE TABLE IF NOT EXISTS {T_MENU} (
    id            BIGINT PRIMARY KEY,
    restaurant_id BIGINT REFERENCES {T_REST}(id),
    menu_order    INT,
    menu_name     TEXT,
    price_text    TEXT,
    price_value   INT,
    is_main       BOOLEAN
);

CREATE TABLE IF NOT EXISTS {T_REMB} (
    id            BIGSERIAL PRIMARY KEY,
    restaurant_id BIGINT REFERENCES {T_REST}(id),
    content       TEXT,
    embedding     VECTOR({EMBED_DIM})
);

-- review_id로 FK: 어느 리뷰의 임베딩인지 추적
CREATE TABLE IF NOT EXISTS {T_VEMB} (
    id        BIGSERIAL PRIMARY KEY,
    review_id BIGINT REFERENCES {T_REV}(id),
    content   TEXT,
    embedding VECTOR({EMBED_DIM})
);

-- 메뉴 임베딩(메뉴 시맨틱 검색용). menu_id + restaurant_id 둘 다 보관.
CREATE TABLE IF NOT EXISTS {T_MEMB} (
    id            BIGSERIAL PRIMARY KEY,
    menu_id       BIGINT REFERENCES {T_MENU}(id),
    restaurant_id BIGINT REFERENCES {T_REST}(id),
    content       TEXT,
    embedding     VECTOR({EMBED_DIM})
);
"""

conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

if RESET_DB:
    # 자식 테이블부터 삭제 (FK 순서). 메뉴도 언어별이라 매 실행 시 함께 재생성.
    drops = [T_MEMB, T_VEMB, T_REMB, T_MENU, T_REV, T_REST]
    for t in drops:
        cur.execute(f"DROP TABLE IF EXISTS {t} CASCADE;")
    print("기존 테이블 삭제:", drops)

cur.execute(DDL)
conn.commit()
cur.close(); conn.close()
print("테이블 생성 완료")

## 4. 일반 테이블 적재 (식당 · 리뷰 · 메뉴)

In [ ]:
REST_COLS = ["id","name","phone","address","postal_code","lat","lng","category","hours",
             "description","image","link","review_count","rating",
             "category_kakao","description_kakao","last_order","kakao_place_id","kakao_place_url",
             "menu_price_min","menu_price_median","menu_count",
             "has_parking","has_group_seating","has_private_room","has_baby_chair",
             "has_kids_menu","allows_pets","has_disabled_access","hours_source"]

def to_int(v):
    return None if v is None or (isinstance(v, float) and np.isnan(v)) else int(v)

conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

# 식당
rows = []
for _, r in rest.iterrows():
    row = [r.get(c) for c in REST_COLS]
    for i, c in enumerate(REST_COLS):
        if c in ("id","review_count","menu_price_min","menu_price_median","menu_count"):
            row[i] = to_int(row[i])
    rows.append(tuple(row))
execute_values(cur,
    f"INSERT INTO {T_REST} ({','.join(REST_COLS)}) VALUES %s ON CONFLICT (id) DO NOTHING", rows)
print(f"{T_REST}: {len(rows):,}행")

# 리뷰
rev_rows = [(to_int(r["id"]), to_int(r["restaurant_id"]), r["rating"], r["content"])
            for _, r in rev.iterrows()]
execute_values(cur,
    f"INSERT INTO {T_REV} (id, restaurant_id, rating, content) VALUES %s ON CONFLICT (id) DO NOTHING",
    rev_rows)
print(f"{T_REV}: {len(rev_rows):,}행")

# 메뉴 (언어별 — 단일 menu_name 컬럼)
m_rows = [(to_int(r["id"]), to_int(r["restaurant_id"]), to_int(r["menu_order"]),
           r["menu_name"], r["price_text"], to_int(r["price_value"]), bool(r["is_main"]))
          for _, r in menu.iterrows()]
execute_values(cur,
    f"""INSERT INTO {T_MENU}
        (id, restaurant_id, menu_order, menu_name, price_text, price_value, is_main)
        VALUES %s ON CONFLICT (id) DO NOTHING""", m_rows)
print(f"{T_MENU}: {len(m_rows):,}행")

conn.commit(); cur.close(); conn.close()
print("일반 테이블 적재 완료")

## 5. 식당 임베딩 텍스트 생성

```
이름 + 카테고리(트립 + 카카오) + 설명(트립 + 카카오) + 대표메뉴 8개
```

카카오 데이터가 없는 식당(277곳)은 기존 정보만으로 만들어집니다 — **검색에서 배제되지 않습니다.**

In [ ]:
CAT_LABEL = "카테고리" if LANG == "ko" else "Category"
MENU_LABEL = "대표메뉴" if LANG == "ko" else "Signature menu"
FALLBACK  = "음식점" if LANG == "ko" else "Restaurant"

# 식당별 대표메뉴 미리 묶어두기 (언어별 menu 테이블의 단일 menu_name 사용)
main_menu = menu[(menu["is_main"] == True) & (menu["menu_order"] <= MENU_TOP_N)]
menu_by_rest = {}
for rid, g in main_menu.sort_values("menu_order").groupby("restaurant_id"):
    names = [str(n).strip() for n in g["menu_name"] if n and str(n).strip()]
    if names:
        menu_by_rest[int(rid)] = names[:MENU_TOP_N]


def build_embed_text(row):
    parts = []
    if row.get("name"):
        parts.append(str(row["name"]).strip())

    cats = [str(c).strip() for c in (row.get("category"), row.get("category_kakao")) if c]
    if cats:
        merged = ", ".join(dict.fromkeys(", ".join(cats).split(", ")))
        parts.append(f"{CAT_LABEL}: {merged}")

    for d in (row.get("description"), row.get("description_kakao")):
        if d and str(d).strip():
            parts.append(str(d).strip())

    names = menu_by_rest.get(int(row["id"]), [])
    if names:
        parts.append(f"{MENU_LABEL}: " + ", ".join(names))

    return ". ".join(parts).strip() or FALLBACK


rest["embed_text"] = rest.apply(build_embed_text, axis=1)

lens = rest["embed_text"].str.len()
print(f"임베딩 텍스트: 평균 {lens.mean():.0f}자, 중앙값 {lens.median():.0f}자, 최대 {lens.max()}자")
print(f"대표메뉴 포함된 식당: {sum(1 for i in rest['id'] if int(i) in menu_by_rest):,}곳")
print("\n--- 샘플 ---")
print(rest["embed_text"].iloc[0][:250])

## 6. 임베딩 생성 함수 (재시도 포함)

In [ ]:
def embed_batch(texts, max_retries=4):
    """배치 임베딩. 실패 시 지수 백오프로 재시도."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.embeddings.create(
                model=EMBED_MODEL, input=texts, dimensions=EMBED_DIM
            )
            return [d.embedding for d in resp.data]
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = 2 ** attempt
            print(f"  재시도 {attempt}/{max_retries} ({e}) — {wait}초 대기")
            time.sleep(wait)


def embed_and_insert(items, table, id_col):
    """items: [(id, text), ...] → 임베딩 후 (id_col, content, embedding) 적재"""
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    total = len(items)
    for i in range(0, total, BATCH):
        chunk = items[i:i + BATCH]
        vecs = embed_batch([t for _, t in chunk])
        rows = [(rid, txt, str(v)) for (rid, txt), v in zip(chunk, vecs)]
        execute_values(cur,
            f"INSERT INTO {table} ({id_col}, content, embedding) VALUES %s", rows)
        conn.commit()
        print(f"  {min(i+BATCH, total):,}/{total:,}", end="\r")
    cur.close(); conn.close()
    print(f"\n{table}: {total:,}개 완료")


def embed_and_insert_menu(items, table):
    """items: [(menu_id, restaurant_id, text), ...]
    → 임베딩 후 (menu_id, restaurant_id, content, embedding) 적재"""
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    total = len(items)
    for i in range(0, total, BATCH):
        chunk = items[i:i + BATCH]
        vecs = embed_batch([t for _, _, t in chunk])
        rows = [(mid, rid, txt, str(v)) for (mid, rid, txt), v in zip(chunk, vecs)]
        execute_values(cur,
            f"INSERT INTO {table} (menu_id, restaurant_id, content, embedding) VALUES %s", rows)
        conn.commit()
        print(f"  {min(i+BATCH, total):,}/{total:,}", end="\r")
    cur.close(); conn.close()
    print(f"\n{table}: {total:,}개 완료")

print("임베딩 함수 준비 완료")

## 7. 식당 임베딩 (1,000개)

In [ ]:
items = [(int(r["id"]), r["embed_text"]) for _, r in rest.iterrows()]
embed_and_insert(items, T_REMB, "restaurant_id")

## 8. 리뷰 임베딩

**`review_id`를 FK로 저장**합니다 (기존 L2에서는 `restaurant_id`였음).

In [ ]:
rev_items = [(int(r["id"]), str(r["content"]).strip())
             for _, r in rev.iterrows()
             if r["content"] and str(r["content"]).strip()]
print(f"리뷰 임베딩 대상: {len(rev_items):,}개")
embed_and_insert(rev_items, T_VEMB, "review_id")

In [ ]:
# ── 메뉴 임베딩 (menu_embedding_{lang}) ──
# 앱의 메뉴 시맨틱 검색이 menu_embedding_{lang} 를 사용한다. 각 메뉴명을 임베딩해
# menu_id + restaurant_id 와 함께 적재한다. (현재 DB: ko 10,003 / en 9,697)
menu_items = [(int(r["id"]), int(r["restaurant_id"]), str(r["menu_name"]).strip())
              for _, r in menu.iterrows()
              if r["menu_name"] and str(r["menu_name"]).strip()]
print(f"메뉴 임베딩 대상: {len(menu_items):,}개")
embed_and_insert_menu(menu_items, T_MEMB)

## 9. 인덱스

In [ ]:
IDX = f"""
CREATE INDEX IF NOT EXISTS idx_{T_REMB}_vec ON {T_REMB}
    USING hnsw (embedding vector_cosine_ops);
CREATE INDEX IF NOT EXISTS idx_{T_VEMB}_vec ON {T_VEMB}
    USING hnsw (embedding vector_cosine_ops);
CREATE INDEX IF NOT EXISTS idx_{T_MEMB}_vec ON {T_MEMB}
    USING hnsw (embedding vector_cosine_ops);

CREATE INDEX IF NOT EXISTS idx_{T_REST}_cat  ON {T_REST} USING gin (category gin_trgm_ops);
CREATE INDEX IF NOT EXISTS idx_{T_REST}_catk ON {T_REST} USING gin (category_kakao gin_trgm_ops);

CREATE INDEX IF NOT EXISTS idx_{T_REV}_rid   ON {T_REV} (restaurant_id);
CREATE INDEX IF NOT EXISTS idx_{T_REMB}_rid  ON {T_REMB} (restaurant_id);
CREATE INDEX IF NOT EXISTS idx_{T_VEMB}_rid  ON {T_VEMB} (review_id);
CREATE INDEX IF NOT EXISTS idx_{T_MENU}_rid  ON {T_MENU} (restaurant_id);
CREATE INDEX IF NOT EXISTS idx_{T_MEMB}_mid  ON {T_MEMB} (menu_id);
CREATE INDEX IF NOT EXISTS idx_{T_MEMB}_rid  ON {T_MEMB} (restaurant_id);
"""
conn = psycopg2.connect(**DB_PARAMS); cur = conn.cursor()
cur.execute(IDX); conn.commit(); cur.close(); conn.close()
print("인덱스 생성 완료")

## 10. 검증

In [ ]:
conn = psycopg2.connect(**DB_PARAMS); cur = conn.cursor()

print("=== 행 수 ===")
for t in [T_REST, T_REV, T_MENU, T_REMB, T_VEMB, T_MEMB]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:26} {cur.fetchone()[0]:,}")

print("\n=== 벡터 차원 ===")
for t in [T_REMB, T_VEMB, T_MEMB]:
    cur.execute(f"SELECT vector_dims(embedding) FROM {t} LIMIT 1")
    row = cur.fetchone()
    print(f"  {t:26} {row[0] if row else 'EMPTY'}")

print("\n=== 카카오 데이터 적재 확인 ===")
cur.execute(f"""SELECT
    COUNT(description_kakao), COUNT(category_kakao),
    COUNT(menu_price_median), COUNT(*) FILTER (WHERE has_parking IS TRUE)
  FROM {T_REST}""")
d, c_, p, pk = cur.fetchone()
print(f"  description_kakao {d} | category_kakao {c_} | 가격통계 {p} | 주차가능 {pk}")

print("\n=== review_embedding FK 확인 ===")
cur.execute(f"""SELECT e.review_id, r.restaurant_id, LEFT(e.content, 30)
               FROM {T_VEMB} e JOIN {T_REV} r ON e.review_id = r.id LIMIT 2""")
for row in cur.fetchall():
    print(f"  review_id={row[0]} → restaurant_id={row[1]} | {row[2]}...")

print("\n=== menu_embedding FK 확인 ===")
cur.execute(f"""SELECT e.menu_id, e.restaurant_id, m.menu_name
               FROM {T_MEMB} e JOIN {T_MENU} m ON e.menu_id = m.id LIMIT 2""")
for row in cur.fetchall():
    print(f"  menu_id={row[0]} → restaurant_id={row[1]} | {row[2]}")

cur.close(); conn.close()
print("\n✅ 검증 완료")